In [1]:
import os

# This forces TensorFlow to use the internal Keras 2 instead of the new Keras 3
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# Natural Language Processing with TensorFlow

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/08-example-nlp-problems.png)
*A handful of example natural language processing (NLP) and natural language understanding (NLU) problems. These are also often referred to as sequence problems (going from one sequence to another).*

The main goal of [natural language processing (NLP)](https://becominghuman.ai/a-simple-introduction-to-natural-language-processing-ea66a1747b32) is to derive information from natural language.

Natural language is a broad term but you can consider it to cover any of the following:
* Text (such as that contained in an email, blog post, book, Tweet)
* Speech (a conversation you have with a doctor, voice commands you give to a smart speaker)

Under the umbrellas of text and speech there are many different things you might want to do.

If you're building an email application, you might want to scan incoming emails to see if they're spam or not spam (classification).

If you're trying to analyse customer feedback complaints, you might want to discover which section of your business they're for.

> 🔑 **Note:** Both of these types of data are often referred to as *sequences* (a sentence is a sequence of words). So a common term you'll come across in NLP problems is called *seq2seq*, in other words, finding information in one sequence to produce another sequence (e.g. converting a speech command to a sequence of text-based steps).

To get hands-on with NLP in TensorFlow, we're going to practice the steps we've used previously but this time with text data:

```
Text -> turn into numbers -> build a model -> train the model to find patterns -> use patterns (make predictions)
```

> 📖 **Resource:** For a great overview of NLP and the different problems within it, read the article [*A Simple Introduction to Natural Language Processing*](https://becominghuman.ai/a-simple-introduction-to-natural-language-processing-ea66a1747b32).

## What we're going to cover

Let's get specific hey?

* Downloading a text dataset
* Visualizing text data
* Converting text into numbers using tokenization
* Turning our tokenized text into an embedding
* Modelling a text dataset
  * Starting with a baseline (TF-IDF)
  * Building several deep learning text models
    * Dense, LSTM, GRU, Conv1D, Transfer learning
* Comparing the performance of each our models
* Combining our models into an ensemble
* Saving and loading a trained model
* Find the most wrong predictions


## Get helper functions

In past modules, we've created a bunch of helper functions to do small tasks required for our notebooks.

Rather than rewrite all of these, we can import a script and load them in from there.

The script containing our helper functions can be [found on GitHub](https://raw.githubusercontent.com/Shailya777/Learning-Logs/refs/heads/main/Deep_Learning_with_Tensorflow/helper_functions.py).

In [2]:
# Run the below lines if helper functions are not already downloaded:
# Downloading Helper Functions:

#import wget
#url = 'https://raw.githubusercontent.com/Shailya777/Learning-Logs/refs/heads/main/Deep_Learning_with_Tensorflow/helper_functions.py'
#file = wget.download(url)

In [4]:
# Importing helper functions from Downloaded Script:
from helper_functions import unzip_data, plot_loss_curves, compare_historys, create_tensorboard_callback

## Download a text dataset

Let's start by download a text dataset. We'll be using the [Real or Not?](https://www.kaggle.com/c/nlp-getting-started/data) dataset from Kaggle which contains text-based Tweets about natural disasters. 

The Real Tweets are actually about disasters, for example:

```
Jetstar and Virgin forced to cancel Bali flights again because of ash from Mount Raung volcano
```

The Not Real Tweets are Tweets not about disasters (they can be on anything), for example:

```
'Education is the most powerful weapon which you can use to change the world.' Nelson #Mandela #quote
```

For convenience, the dataset has been [downloaded from Kaggle](https://www.kaggle.com/c/nlp-getting-started/data) (doing this requires a Kaggle account) and uploaded as a downloadable zip file. 

Note: The original downloaded data has not been altered to how you would download it from Kaggle.

In [5]:
# Run below lines of code if data is not already downloaded:
# Downloading Text Data:
#url = 'https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip'
#file = wget.download(url)

# Unzipping The Data:
#unzip_data('nlp_getting_started.zip')

Unzipping `nlp_getting_started.zip` gives the following 3 `.csv` files:
* `sample_submission.csv` - an example of the file you'd submit to the Kaggle competition of your model's predictions.
* `train.csv` - training samples of real and not real diaster Tweets.
* `test.csv` - testing samples of real and not real diaster Tweets.

## Visualizing a text dataset

Once you've acquired a new dataset to work with, what should you do first?

Explore it? Inspect it? Verify it? Become one with it?

All correct.

Right now, our text data samples are in the form of `.csv` files. For an easy way to make them visual, let's turn them into pandas DataFrame's.

> 📖 **Reading:** You might come across text datasets in many different formats. Aside from CSV files (what we're working with), you'll probably encounter `.txt` files and `.json` files too. For working with these type of files, I'd recommend reading the two following articles by RealPython:
* [How to Read and Write Files in Python](https://realpython.com/read-write-files-python/)
* [Working with JSON Data in Python](https://realpython.com/python-json/)

In [6]:
# turning .csv Data into Pandas Dataframe:

import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [7]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [8]:
# Shuffling Training Data:
train_df_shuffled = train_df.sample(frac= 1,
                                   random_state= 42)

In [9]:
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


Notice how the training data has a `"target"` column.

We're going to be writing code to find patterns (e.g. different combinations of words) in the `"text"` column of the training dataset to predict the value of the `"target"` column.

The test dataset doesn't have a `"target"` column.

```
Inputs (text column) -> Machine Learning Algorithm -> Outputs (target column)
```

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/08-text-classification-inputs-and-outputs.png)
*Example text classification inputs and outputs for the problem of classifying whether a Tweet is about a disaster or not.*

In [10]:
# Testing Data:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [11]:
# How many examples of each class are in Training Data:
train_df_shuffled['target'].value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

Since we have two target values, we're dealing with a **binary classification** problem.

It's fairly balanced too, about 60% negative class (`target = 0`) and 40% positive class (`target = 1`).

Where, 

* `1` = a real disaster Tweet
* `0` = not a real disaster Tweet

In [12]:
# Total Number of Samples in Data:
print(f'Training Data Samples: {len(train_df_shuffled)}')
print(f'Test Data Samples: {len(test_df)}')
print(f'Total Number of Samples: {len(train_df_shuffled) + len(test_df)}')

Training Data Samples: 7613
Test Data Samples: 3263
Total Number of Samples: 10876


Alright, seems like we've got a decent amount of training and test data. If anything, we've got an abundance of testing examples, usually a split of 90/10 (90% training, 10% testing) or 80/20 is suffice.

Okay, time to visualize, let's write some code to visualize random text samples.

Question: Why visualize random samples? You could visualize samples in order but this could lead to only seeing a certain subset of data. Better to visualize a substantial quantity (100+) of random samples to get an idea of the different kinds of data you're working with. In machine learning, never underestimate the power of randomness.

In [13]:
# Visualizing 10 Random Training Samples:

import random
random_index = random.randint(0, len(train_df) - 10) # create random indexes not higher than the total number of samples

for row in train_df_shuffled[['text', 'target']][random_index:random_index+10].itertuples():
    _, text, target = row
    print(f'Target: {target}', '(Real Disaster)' if target > 0 else '(Not Real Disaster)')
    print(f'Text:\n {text}\n')

Target: 1 (Real Disaster)
Text:
 Whoa a #wmata train derailed at Smithsonian?

Target: 1 (Real Disaster)
Text:
 Rescuers find survivors of Nepal earthquake buried ... http://t.co/I8SJ1KWs1D

Target: 1 (Real Disaster)
Text:
 #Kisii Police in Kisii hunt for students over failed arson plot: Police in Kisii hunt for students... http://t.co/5bdrFU1duo #CountyNews

Target: 1 (Real Disaster)
Text:
 Debris confirmed from MH370; relatives hope for discovery of crash site http://t.co/rLFtjmHHvT via @Reuters #Video

Target: 1 (Real Disaster)
Text:
 @HopefulBatgirl went down I was beaten and blown up. Then next thing I know Ra Al Ghul brought me back to life and I escaped and for a---

Target: 0 (Not Real Disaster)
Text:
 @_charleyisqueen Yeah well maybe if the barber didn't cut my hair too short on top I wouldn't of gone through the egg trouble????

Target: 0 (Not Real Disaster)
Text:
 lmfao fucking luis hhahaha im dead

Target: 0 (Not Real Disaster)
Text:
 Our garbage truck really caught on fire

### Split data into training and validation sets

Since the test set has no labels and we need a way to evalaute our trained models, we'll split off some of the training data and create a validation set.

When our model trains (tries patterns in the Tweet samples), it'll only see data from the training set and we can see how it performs on unseen data using the validation set.

We'll convert our splits from pandas Series datatypes to lists of strings (for the text) and lists of ints (for the labels) for ease of use later.

To split our training dataset and create a validation dataset, we'll use Scikit-Learn's [`train_test_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) method and dedicate 10% of the training samples to the validation set.

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
# Use Train Test Split to Split Training Data into Training and Validation Sets:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled['text'].to_numpy(),
                                                                           train_df_shuffled['target'].to_numpy(),
                                                                           test_size= 0.1,
                                                                           random_state= 42)

In [16]:
# Checking Lengths:
len(train_sentences), len(train_labels), len(val_sentences), len(val_labels)

(6851, 6851, 762, 762)

In [17]:
# Checking First 10 Examples in training sentences and it's labels:
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

## Converting text into numbers

We've got a training set and a validation set containing Tweets and labels.

Our labels are in numerical form (`0` and `1`) but our Tweets are in string form.

Question: What do you think we have to do before we can use a machine learning algorithm with our text data? 

If you answered something along the lines of "turn it into numbers", you're correct. A machine learning algorithm requires its inputs to be in numerical form.

In NLP, there are two main concepts for turning text into numbers:
* **Tokenization** - A straight mapping from word or character or sub-word to a numerical value. There are three main levels of tokenization:
  1. Using **word-level tokenization** with the sentence "I love TensorFlow" might result in "I" being `0`, "love" being `1` and "TensorFlow" being `2`. In this case, every word in a sequence considered a single **token**.
  2. **Character-level tokenization**, such as converting the letters A-Z to values `1-26`. In this case, every character in a sequence considered a single **token**.
  3. **Sub-word tokenization** is in between word-level and character-level tokenization. It involves breaking invidual words into smaller parts and then converting those smaller parts into numbers. For example, "my favourite food is pineapple pizza" might become "my, fav, avour, rite, fo, oo, od, is, pin, ine, app, le, piz, za". After doing this, these sub-words would then be mapped to a numerical value. In this case, every word could be considered multiple **tokens**.
* **Embeddings** - An embedding is a representation of natural language which can be learned. Representation comes in the form of a **feature vector**. For example, the word "dance" could be represented by the 5-dimensional vector `[-0.8547, 0.4559, -0.3332, 0.9877, 0.1112]`. It's important to note here, the size of the feature vector is tuneable. There are two ways to use embeddings: 
  1. **Create your own embedding** - Once your text has been turned into numbers (required for an embedding), you can put them through an embedding layer (such as [`tf.keras.layers.Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding)) and an embedding representation will be learned during model training.
  2. **Reuse a pre-learned embedding** - Many pre-trained embeddings exist online. These pre-trained embeddings have often been learned on large corpuses of text (such as all of Wikipedia) and thus have a good underlying representation of natural language. You can use a pre-trained embedding to initialize your model and fine-tune it to your own specific task.

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/08-tokenization-vs-embedding.png)
*Example of **tokenization** (straight mapping from word to number) and **embedding** (richer representation of relationships between tokens).*

Question: What level of tokenzation should I use? What embedding should should I choose?

It depends on your problem. You could try character-level tokenization/embeddings and word-level tokenization/embeddings and see which perform best. You might even want to try stacking them (e.g. combining the outputs of your embedding layers using [`tf.keras.layers.concatenate`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/concatenate)). 

If you're looking for pre-trained word embeddings, [Word2vec embeddings](http://jalammar.github.io/illustrated-word2vec/), [GloVe embeddings](https://nlp.stanford.edu/projects/glove/) and many of the options available on [TensorFlow Hub](https://tfhub.dev/s?module-type=text-embedding) are great places to start.

Note: Much like searching for a pre-trained computer vision model, you can search for pre-trained word embeddings to use for your problem. Try searching for something like "use pre-trained word embeddings in TensorFlow".

### Text vectorization (tokenization)


We'll practice tokenzation (mapping our words to numbers) first.

To tokenize our words, we'll use the helpful preprocessing layer [`tf.keras.layers.TextVectorization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization).

The `TextVectorization` layer takes the following parameters:
* `max_tokens` - The maximum number of words in your vocabulary (e.g. 20000 or the number of unique words in your text), includes a value for OOV (out of vocabulary) tokens. 
* `standardize` - Method for standardizing text. Default is `"lower_and_strip_punctuation"` which lowers text and removes all punctuation marks.
* `split` - How to split text, default is `"whitespace"` which splits on spaces.
* `ngrams` - How many words to contain per token split, for example, `ngrams=2` splits tokens into continuous sequences of 2.
* `output_mode` -  How to output tokens, can be `"int"` (integer mapping), `"binary"` (one-hot encoding), `"count"` or `"tf-idf"`. See documentation for more.
* `output_sequence_length` - Length of tokenized sequence to output. For example, if `output_sequence_length=150`, all tokenized sequences will be 150 tokens long.
* `pad_to_max_tokens` - Defaults to `False`, if `True`, the output feature axis will be padded to `max_tokens` even if the number of unique tokens in the vocabulary is less than `max_tokens`. Only valid in certain modes, see docs for more.


In [18]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [20]:
# Using Default Parameters for TextVectorization:

text_vectorizer = TextVectorization(max_tokens= None, # how many words in the vocabulary (all of the different words in your text)
                                   standardize= 'lower_and_strip_punctuation', # how to process text
                                   split= 'whitespace', # how to split tokens
                                   ngrams= None, # create groups of n-words?
                                   output_mode= 'int', # how to map tokens to numbers
                                   output_sequence_length= None) # how long should the output sequence of tokens be?
                                   # pad_to_max_tokens= True) # Not valid if using max_tokens=None

We've initialized a `TextVectorization` object with the default settings but let's customize it a little bit for our own use case.

In particular, let's set values for `max_tokens` and `output_sequence_length`.

For `max_tokens` (the number of words in the vocabulary), multiples of 10,000 (`10,000`, `20,000`, `30,000`) or the exact number of unique words in your text (e.g. `32,179`) are common values.

For our use case, we'll use `10,000`.

And for the `output_sequence_length` we'll use the average number of tokens per Tweet in the training set. But first, we'll need to find it.

In [21]:
train_sentences[0]

'@mogacola @zamtriossu i screamed after hitting tweet'

In [22]:
train_sentences[0].split()

['@mogacola', '@zamtriossu', 'i', 'screamed', 'after', 'hitting', 'tweet']

In [23]:
len(train_sentences[0].split())

7

In [24]:
# Average Number of Tokens(Words) in Training Tweets:
round(sum([len(i.split()) for i in train_sentences]) / len(train_sentences))

15

In [25]:
# Setting up TextVectorization with Custom Parameter values:

max_vocab_length = 10000 # max number of words to have in our vocabulary
max_lengths = 15 # max length our sequences will be (e.g. how many words from a Tweet does our model see?)

text_vectorizer = TextVectorization(max_tokens= max_vocab_length,
                                   output_mode= 'int',
                                   output_sequence_length= max_lengths)

To map our `TextVectorization` instance `text_vectorizer` to our data, we can call the `adapt()` method on it whilst passing it our training text.

In [26]:
# Fir the TextVectorizer to Training Data:
text_vectorizer.adapt(train_sentences)

In [27]:
# Creating a Small Sentence and Vectorize it:
sample_sentence = "There's a flood in my street!"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[264,   3, 232,   4,  13, 698,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

It seems we've got a way to turn our text into numbers (in this case, word-level tokenization). Notice the 0's at the end of the returned tensor, this is because we set `output_sequence_length=15`, meaning no matter the size of the sequence we pass to `text_vectorizer`, it always returns a sequence with a length of 15.

How about we try our `text_vectorizer` on a few random sentences?

In [28]:
# Chose a Random Sentence from Training Data and Vectorize it:
random_sentence = random.choice(train_sentences)
print(f'Original Text:\n {random_sentence} \n\n Vectorized Version: {text_vectorizer([random_sentence])}')

Original Text:
 DC Media: tip for getting updates on the derailed #WMATA train today - ask them about movie night and quickly change the subject! 

 Vectorized Version: [[1306  620 7545   10  209 1454   11    2  464 1567  119  124 1554   93
    54]]


We can check the unique tokens in our vocabulary using the `get_vocabulary()` method.

In [29]:
# Get the Unique Words in Vocabulary of Text Vectorizer:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]
print(f'Number of Words in Vocabulary: {len(words_in_vocab)}')
print(f'Top 5 Most Common Words: {top_5_words}')
print(f'Bottom 5 Least Common Words: {bottom_5_words}')

Number of Words in Vocabulary: 10000
Top 5 Most Common Words: ['', '[UNK]', 'the', 'a', 'in']
Bottom 5 Least Common Words: ['pages', 'paeds', 'pads', 'padres', 'paddytomlinson1']


### Creating an Embedding using an Embedding Layer

We've got a way to map our text to numbers. How about we go a step further and turn those numbers into an embedding?

The powerful thing about an embedding is it can be learned during training. This means rather than just being static (e.g. `1` = I, `2` = love, `3` = TensorFlow), a word's numeric representation can be improved as a model goes through data samples.

We can see what an embedding of a word looks like by using the [`tf.keras.layers.Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) layer. 

The main parameters we're concerned about here are:
* `input_dim` - The size of the vocabulary (e.g. `len(text_vectorizer.get_vocabulary()`).
* `output_dim` - The size of the output embedding vector, for example, a value of `100` outputs a  feature vector of size 100 for each word.
* `embeddings_initializer` - How to initialize the embeddings matrix, default is `"uniform"` which randomly initalizes embedding matrix with uniform distribution. This can be changed for using pre-learned embeddings.
* `input_length` - Length of sequences being passed to embedding layer.

In [30]:
from tensorflow.keras.layers import Embedding

In [31]:
tf.random.set_seed(42)

embedding = Embedding(input_dim= max_vocab_length, # set input shape
                     output_dim= 128, # set size of embedding vector
                     embeddings_initializer= 'uniform', # default, intialize randomly
                     name= 'Embedding_1')

In [32]:
embedding

In [33]:
# getting a Random Sentence from Training Sentences to Embed:

random_sentence = random.choice(train_sentences)
print(f'Original Text:\n {random_sentence}')
print(f'Embedded Version:\n')

# Embed The Random Sentence (Turn it into Numerical Representation (into Dense Vectors of Fixed Size)):
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

Original Text:
 @KalinAndMyles @KalinWhite my ig is being blown up just with hackers I need it to stop #givebackkalinwhiteaccount
Embedded Version:



<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.0430898 , -0.01179854,  0.00906976, ...,  0.01913322,
         -0.0342361 , -0.03157746],
        [ 0.0430898 , -0.01179854,  0.00906976, ...,  0.01913322,
         -0.0342361 , -0.03157746],
        [ 0.02067946,  0.04830356, -0.04294548, ..., -0.02807599,
          0.01306964, -0.00763386],
        ...,
        [ 0.02187512, -0.004135  , -0.01430256, ...,  0.00030886,
          0.00170923,  0.01014311],
        [-0.03382337, -0.04197621, -0.00748562, ..., -0.01490899,
         -0.04639475,  0.02209374],
        [ 0.03339528,  0.02069496, -0.03515862, ...,  0.01056272,
          0.03224167,  0.00877015]]], dtype=float32)>

In [35]:
# Checking Out Single Token (word)'s Embedding:
sample_embed[0][0]

<tf.Tensor: shape=(128,), dtype=float32, numpy=
array([ 4.30897959e-02, -1.17985383e-02,  9.06976312e-03, -3.91230211e-02,
       -3.81620526e-02, -1.55854002e-02, -4.94340807e-03,  3.49655785e-02,
        4.61283587e-02, -4.47987579e-02, -2.20915079e-02,  1.52433999e-02,
        2.81444080e-02, -6.38142228e-05, -2.98318751e-02, -2.62936000e-02,
        8.99929926e-03,  4.86075878e-04,  4.09621857e-02, -4.18040268e-02,
       -3.42477337e-02,  4.97557260e-02,  2.24756040e-02,  3.50792333e-03,
        4.70077731e-02,  4.65415232e-02,  4.57524396e-02, -2.90528778e-02,
        4.15655114e-02, -3.74020264e-03, -1.34271272e-02, -4.87786904e-02,
       -2.78558582e-03,  1.90679096e-02, -2.30773538e-03,  3.55191156e-03,
        4.93324883e-02, -1.58895254e-02, -1.80405863e-02,  4.26856019e-02,
       -2.00557951e-02, -4.10643928e-02, -1.48619413e-02, -2.15341449e-02,
       -3.40523571e-03, -5.50828129e-03, -2.07414869e-02, -3.14100385e-02,
        2.44742371e-02, -1.71609893e-02, -4.62993868

In [36]:
random_sentence.split()[0]

'@KalinAndMyles'